# Adversarial Validation — Merged HAR Dataset Leakage Check

The merged dataset (`data/merged/har_merged.npz`, built by `har_merge.ipynb` from
UCI HAR + WISDM + MotionSense + HHAR) gives suspiciously high results
(CNN baseline ≈ **0.97** test balanced-accuracy / macro-F1). This notebook checks whether
that is **data leakage** using *adversarial validation* plus targeted leakage probes, and
quantifies the impact by retraining under a leakage-free split.

**Suspected leakage mechanisms**

1. **Random, ungrouped split.** `src/merged_dataset.py::load_and_prepare_data` uses
   `StratifiedShuffleSplit(test_size=0.15, random_state=42)` on `(X, y)` only — the *same
   subject's* windows land in both train and test.
2. **Overlapping sliding windows.** Windows use `WINDOW=128, STEP=64` (50% overlap) within each
   `(source, subject, activity)` run, so **consecutive array indices are near-duplicates** (share
   64/128 samples). A random split scatters adjacent near-duplicates across train/test.
3. **UCI original train+test merged then re-split**, destroying the official held-out boundary.

**What each section does**

| § | Check | Leakage signal |
|---|-------|----------------|
| 4 | Adversarial validation (train-vs-test AUC) | AUC ≈ 0.5 ⇒ test indistinguishable from train ⇒ no real held-out gap |
| 5 | Source separability + subject overlap | sources trivially separable; ~100% of test subjects also in train |
| 6 | Near-duplicate / overlapping-window count | many test windows have an overlapping twin in train |
| 7 | Retrain: random vs subject-grouped split | large balanced-accuracy drop = quantified inflation |
| 8 | Verdict + recommended fix | |

> **Prerequisite:** run `notebooks/har_merge.ipynb` end-to-end first to produce
> `data/merged/har_merged.npz`. This notebook errors clearly if it is missing.

## 1. Setup & robust load

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Resolve the repo root whether the kernel cwd is the repo root or notebooks/.
cwd = Path.cwd()
REPO_ROOT = cwd if (cwd / "configs").exists() else cwd.parent
assert (REPO_ROOT / "configs").exists(), f"Could not locate repo root from {cwd}"

# Make src/ importable (sibling modules import each other by bare name).
SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from config_loader import ConfigLoader
from baseline_cnn import set_seed, LABEL_NAMES

set_seed(42)
print("Repo root:", REPO_ROOT)
print("Labels:", LABEL_NAMES)

In [ ]:
# Load the SAME merged config the training pipeline uses, so split ratios / seed match.
config = ConfigLoader(config_dir=REPO_ROOT / "configs").load_experiment("har_merged", "baseline_cnn")

SEED = config["training"].get("seed", 42)
TEST_RATIO = config["dataset"]["split"]["test_ratio"]
VAL_RATIO = config["dataset"]["split"]["val_ratio"]

# Resolve the npz path off the repo root (config stores "./data/merged/har_merged.npz").
npz_path = (REPO_ROOT / config["dataset"]["paths"]["root"]).resolve()

if not npz_path.exists():
    raise FileNotFoundError(
        f"Merged dataset not found at {npz_path}.\n"
        "Run notebooks/har_merge.ipynb end-to-end first to generate har_merged.npz."
    )

data = np.load(npz_path, allow_pickle=True)
expected = {"X", "y", "activity", "subject", "source"}
missing = expected - set(data.files)
if missing:
    raise KeyError(
        f"{npz_path} is missing arrays {sorted(missing)} (found {sorted(data.files)}). "
        "Re-run har_merge.ipynb; the diagnostic needs subject/source metadata."
    )

X = data["X"].astype(np.float32)            # (N, 128, 3)
y = data["y"].astype(np.int64)              # (N,)
activity = data["activity"].astype(str)     # (N,)
subject = data["subject"].astype(str)       # (N,)  e.g. "uci_1", "wisdm_12"
source = data["source"].astype(str)         # (N,)  in {uci, wisdm, motionsense, hhar}

N = len(y)
print(f"Loaded {N:,} windows of shape {X.shape[1:]} from {npz_path.name}")
print("Seed / test_ratio / val_ratio:", SEED, TEST_RATIO, VAL_RATIO)
print("Sources:", sorted(pd.unique(source)))
print("Subjects:", len(pd.unique(subject)), "| Classes:", sorted(pd.unique(y)))

## 2. Reproduce the exact split used in training

We replicate the two-stage `StratifiedShuffleSplit` from
`merged_dataset.py::load_and_prepare_data` with the same seed and ratios, recovering the
**integer positions** into the original arrays — so `subject[test_idx]`, `source[train_idx]`, etc.
stay aligned with the same rows the model actually trained / tested on.

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit

# Stage 1: carve out the test set (15%).
sss_test = StratifiedShuffleSplit(n_splits=1, test_size=TEST_RATIO, random_state=SEED)
trainval_idx, test_idx = next(sss_test.split(X, y))

# Stage 2: carve val out of the remaining trainval (same formula as the loader).
val_share = VAL_RATIO / (1.0 - TEST_RATIO)
sss_val = StratifiedShuffleSplit(n_splits=1, test_size=val_share, random_state=SEED)
train_rel, val_rel = next(sss_val.split(X[trainval_idx], y[trainval_idx]))
train_idx = trainval_idx[train_rel]
val_idx = trainval_idx[val_rel]

# Binary membership for adversarial validation; val folded into the "train side".
is_test = np.zeros(N, dtype=bool)
is_test[test_idx] = True
train_side_idx = np.concatenate([train_idx, val_idx])

print(f"train={len(train_idx):,}  val={len(val_idx):,}  test={len(test_idx):,}")
assert len(set(train_idx) & set(test_idx)) == 0
assert len(train_idx) + len(val_idx) + len(test_idx) == N

## 3. Feature extraction

A cheap, vectorized per-window feature representation (per-axis + magnitude statistics, mean-abs
first difference, pairwise axis correlations). Used both by the adversarial classifier and the
random-vs-grouped retraining comparison. Near-duplicate overlapping windows produce nearly
identical features — exactly what the adversarial classifier would latch onto.

In [ ]:
def extract_features(X):
    # Vectorized per-window features. X: (N, T, 3) -> (N, F) float32.
    X = X.astype(np.float64)
    axes = ["x", "y", "z"]
    cols, names = [], []

    per_axis = {
        "mean": X.mean(axis=1),
        "std": X.std(axis=1),
        "min": X.min(axis=1),
        "max": X.max(axis=1),
        "median": np.median(X, axis=1),
        "p25": np.percentile(X, 25, axis=1),
        "p75": np.percentile(X, 75, axis=1),
        "ptp": X.max(axis=1) - X.min(axis=1),
        "mabs": np.abs(X).mean(axis=1),
        "energy": (X ** 2).mean(axis=1),
        "dmabs": np.abs(np.diff(X, axis=1)).mean(axis=1),  # mean-abs first difference
    }
    for label, arr in per_axis.items():
        for i, ax in enumerate(axes):
            cols.append(arr[:, i]); names.append(f"{label}_{ax}")

    mag = np.sqrt((X ** 2).sum(axis=2))  # (N, T)
    for label, arr in [("mean", mag.mean(axis=1)), ("std", mag.std(axis=1)),
                       ("min", mag.min(axis=1)), ("max", mag.max(axis=1)),
                       ("median", np.median(mag, axis=1)), ("energy", (mag ** 2).mean(axis=1))]:
        cols.append(arr); names.append(f"mag_{label}")

    Xc = X - X.mean(axis=1, keepdims=True)
    def corr(i, j):
        num = (Xc[:, :, i] * Xc[:, :, j]).sum(axis=1)
        den = np.sqrt((Xc[:, :, i] ** 2).sum(axis=1) * (Xc[:, :, j] ** 2).sum(axis=1)) + 1e-12
        return num / den
    for (i, j), nm in [((0, 1), "corr_xy"), ((0, 2), "corr_xz"), ((1, 2), "corr_yz")]:
        cols.append(corr(i, j)); names.append(nm)

    return np.column_stack(cols).astype(np.float32), names


Xf, feat_names = extract_features(X)
print("Feature matrix:", Xf.shape, "| n_features:", len(feat_names))

## 4. Adversarial validation — can a model tell *train* from *test*?

We train a classifier whose target is "is this window in the test split?" and measure
cross-validated ROC-AUC.

**Interpretation.** With a *proper* held-out split, train and test differ enough that AUC > 0.5.
Here we expect **AUC ≈ 0.5** — train and test are statistically indistinguishable because the
random split draws both from the same pool of subjects and overlapping windows. That means the test
set offers *no genuine generalization gap*, so the ~0.97 score is optimistic. The probes in §5–6
explain *why*.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold, train_test_split
from sklearn.inspection import permutation_importance

cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
m = is_test.astype(int)

adv_auc = cross_val_score(
    HistGradientBoostingClassifier(random_state=SEED),
    Xf, m, cv=cv5, scoring="roc_auc", n_jobs=-1,
)
print(f"Adversarial (train-vs-test) ROC-AUC: {adv_auc.mean():.4f} ± {adv_auc.std():.4f}")
if adv_auc.mean() < 0.55:
    print("-> ~0.5: train and test are indistinguishable. No real held-out gap (leakage symptom).")
else:
    print("-> >0.55: a genuine distribution shift exists between the splits.")

In [ ]:
# Permutation importances on a clean holdout of the adversarial task.
Xtr, Xte, mtr, mte = train_test_split(Xf, m, test_size=0.3, stratify=m, random_state=SEED)
adv_clf = HistGradientBoostingClassifier(random_state=SEED).fit(Xtr, mtr)
perm = permutation_importance(adv_clf, Xte, mte, n_repeats=5, random_state=SEED,
                              scoring="roc_auc", n_jobs=-1)
order = np.argsort(perm.importances_mean)[::-1][:15]

fig, ax = plt.subplots(figsize=(7, 5))
ax.barh([feat_names[i] for i in order][::-1], perm.importances_mean[order][::-1], color="#4c72b0")
ax.set_xlabel("Permutation importance (ROC-AUC drop)")
ax.set_title("Top features discriminating train vs test\n(flat/near-zero ⇒ splits indistinguishable)")
fig.tight_layout()
plt.show()
print("Max single-feature importance:", float(perm.importances_mean.max()))

## 5. Source separability & subject overlap

Two complementary probes:

* **Source separability** — can a model tell which of the 4 datasets a window came from? If yes
  (AUC ≈ 1.0), each source has a strong signature; mixing them without grouping lets a model
  exploit source-specific shortcuts.
* **Subject overlap** — what fraction of *test* windows belong to a subject that also appears in
  *train*? With a subject-grouped split this would be 0; here we expect ≈ 1.0. Subject IDs are
  source-prefixed (`uci_1` vs `wisdm_1`), so set operations are collision-safe across sources.

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import ConfusionMatrixDisplay

src_codes = LabelEncoder().fit(source)
src_y = src_codes.transform(source)
src_labels = list(src_codes.classes_)

src_auc = cross_val_score(
    HistGradientBoostingClassifier(random_state=SEED),
    Xf, src_y, cv=cv5, scoring="roc_auc_ovr", n_jobs=-1,
)
print(f"Source-identification ROC-AUC (OVR): {src_auc.mean():.4f} ± {src_auc.std():.4f}")

src_pred = cross_val_predict(HistGradientBoostingClassifier(random_state=SEED),
                             Xf, src_y, cv=cv5, n_jobs=-1)
fig, ax = plt.subplots(figsize=(5.5, 5))
ConfusionMatrixDisplay.from_predictions(src_y, src_pred, display_labels=src_labels,
                                        normalize="true", ax=ax, colorbar=False,
                                        xticks_rotation=45, values_format=".2f")
ax.set_title("Source prediction (normalized)")
fig.tight_layout()
plt.show()

In [ ]:
# Subject overlap between the train side (train+val) and test.
train_subj = set(subject[train_side_idx])
test_subj = set(subject[test_idx])
shared = train_subj & test_subj
test_window_overlap = np.isin(subject[test_idx], list(train_subj)).mean()

print(f"Subjects on train side: {len(train_subj)} | in test: {len(test_subj)} | shared: {len(shared)}")
print(f"Fraction of TEST WINDOWS whose subject also appears in train: {test_window_overlap:.4f}")

rows = []
for s in sorted(pd.unique(source)):
    tr_s = set(subject[train_side_idx][source[train_side_idx] == s])
    te_idx_s = test_idx[source[test_idx] == s]
    te_s = set(subject[te_idx_s])
    frac = np.isin(subject[te_idx_s], list(tr_s)).mean() if len(te_idx_s) else float("nan")
    rows.append({"source": s, "train_subjects": len(tr_s), "test_subjects": len(te_s),
                 "shared_subjects": len(tr_s & te_s), "test_window_overlap": round(float(frac), 4)})
overlap_df = pd.DataFrame(rows)
overlap_df

## 6. Near-duplicate / overlapping-window leakage

Windows are built per contiguous `(source, subject, activity)` run with 50% overlap, so window `i`
and window `i±1` in the *same run* share 64/128 samples. We count how many **test** windows have an
adjacent overlapping twin assigned to the **train side**, and corroborate with the correlation
between such pairs (expected median ≳ 0.95).

In [ ]:
# Group key over original ordering; only same-key neighbors are overlapping twins.
key = pd.factorize(pd.Series(source) + "|" + pd.Series(subject) + "|" + pd.Series(activity))[0]
on_train_side = ~is_test

same_prev = np.zeros(N, dtype=bool); same_prev[1:] = key[1:] == key[:-1]
same_next = np.zeros(N, dtype=bool); same_next[:-1] = key[:-1] == key[1:]
prev_train = np.zeros(N, dtype=bool); prev_train[1:] = on_train_side[:-1]
next_train = np.zeros(N, dtype=bool); next_train[:-1] = on_train_side[1:]

has_adj_train_dup = (same_prev & prev_train) | (same_next & next_train)
dup_frac = has_adj_train_dup[is_test].mean()
print(f"Test windows with an adjacent overlapping twin on the train side: "
      f"{int(has_adj_train_dup[is_test].sum()):,} / {is_test.sum():,}  ({dup_frac:.4f})")

In [ ]:
# Correlation between each such test window and its overlapping train neighbor.
rng = np.random.default_rng(SEED)
test_positions = np.where(is_test)[0]
cands = test_positions[has_adj_train_dup[test_positions]]
sample = rng.choice(cands, size=min(800, len(cands)), replace=False) if len(cands) else np.array([], int)

corrs = []
for i in sample:
    j = i - 1 if (same_prev[i] and prev_train[i]) else i + 1
    a, b = X[i].ravel(), X[j].ravel()
    corrs.append(np.corrcoef(a, b)[0, 1])
corrs = np.array(corrs)

if len(corrs):
    print(f"Neighbor-pair correlation: median={np.median(corrs):.3f}  "
          f"mean={corrs.mean():.3f}  >0.9: {(corrs > 0.9).mean():.3f}")
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.hist(corrs, bins=40, color="#c44e52")
    ax.set_xlabel("corr(test window, overlapping train neighbor)")
    ax.set_ylabel("count")
    ax.set_title("Near-duplicate leakage: test↔train overlapping pairs")
    fig.tight_layout(); plt.show()
else:
    print("No adjacent overlapping test/train pairs found.")

## 7. Decisive experiment — random vs subject-grouped split

If the high score comes from leakage, it should **collapse** when we forbid a subject from appearing
in both train and test. We compare a fast `HistGradientBoostingClassifier` (on the §3 features) under:

* **(a) random split** — the current `StratifiedShuffleSplit` (should reproduce a high score), and
* **(b) subject-grouped split** — `StratifiedGroupKFold` grouped by `subject` (leakage-free), plus
* **(c) leave-one-dataset-out** by `source` (worst-case cross-domain).

The gap (a) − (b) is the leakage magnitude. The signal is representation-agnostic, so the tree-model
gap mirrors the CNN's. An optional CNN re-run is provided behind `RUN_CNN` for direct corroboration.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold, LeaveOneGroupOut
from sklearn.metrics import balanced_accuracy_score, f1_score

def fit_eval(tr, te):
    clf = HistGradientBoostingClassifier(random_state=SEED).fit(Xf[tr], y[tr])
    p = clf.predict(Xf[te])
    return balanced_accuracy_score(y[te], p), f1_score(y[te], p, average="macro")

# (a) Current random split: train on train_idx, evaluate on the held-out test_idx.
ba_rand, f1_rand = fit_eval(train_idx, test_idx)
print(f"(a) random split        bal-acc={ba_rand:.4f}  macro-F1={f1_rand:.4f}")

# (b) Subject-grouped, class-stratified 5-fold.
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
g_ba, g_f1 = [], []
for tr, te in sgkf.split(Xf, y, groups=subject):
    ba, f1 = fit_eval(tr, te); g_ba.append(ba); g_f1.append(f1)
ba_group, f1_group = float(np.mean(g_ba)), float(np.mean(g_f1))
print(f"(b) subject-grouped     bal-acc={ba_group:.4f}  macro-F1={f1_group:.4f}  (5-fold mean)")

# (c) Leave-one-dataset-out by source.
logo = LeaveOneGroupOut()
lodo_rows = []
for tr, te in logo.split(Xf, y, groups=source):
    held = source[te][0]
    ba, f1 = fit_eval(tr, te)
    lodo_rows.append({"held_out_source": held, "bal_acc": round(ba, 4), "macro_f1": round(f1, 4)})
lodo_df = pd.DataFrame(lodo_rows)
ba_lodo = float(lodo_df["bal_acc"].mean())

print(f"\nLeakage gap (random - subject-grouped) bal-acc: {ba_rand - ba_group:+.4f}")
lodo_df

In [ ]:
# Summary comparison plot.
labels = ["random\n(leaky)", "subject-grouped\n(leak-free)", "leave-one-source-out"]
ba_vals = [ba_rand, ba_group, ba_lodo]
f1_vals = [f1_rand, f1_group, float(lodo_df["macro_f1"].mean())]

x = np.arange(len(labels)); w = 0.38
fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.bar(x - w/2, ba_vals, w, label="balanced accuracy", color="#4c72b0")
ax.bar(x + w/2, f1_vals, w, label="macro F1", color="#dd8452")
ax.axhline(0.2, ls="--", c="grey", lw=1, label="chance (5 classes)")
for xi, v in zip(x - w/2, ba_vals):
    ax.text(xi, v + 0.01, f"{v:.2f}", ha="center", fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylim(0, 1.05); ax.set_ylabel("score")
ax.set_title("Performance under random vs leakage-free splits")
ax.legend(); fig.tight_layout(); plt.show()

In [ ]:
# Optional CNN corroboration (slow). Reuses the project's CNNClassifier.
RUN_CNN = False  # set True to train the actual baseline CNN under both splits

if RUN_CNN:
    import torch
    from torch.utils.data import DataLoader
    from sklearn.model_selection import GroupShuffleSplit
    from baseline_cnn import build_model
    from merged_dataset import MergedHARDataset

    device = "cuda" if torch.cuda.is_available() else "cpu"
    EPOCHS = 15  # shortened for a quick comparison

    def quick_cnn(tr_idx, te_idx):
        train_ds = MergedHARDataset(X[tr_idx], y[tr_idx])
        test_ds = MergedHARDataset(X[te_idx], y[te_idx],
                                   norm_mean=train_ds.norm_mean, norm_std=train_ds.norm_std)
        tl = DataLoader(train_ds, batch_size=config["training"]["batch_size"], shuffle=True)
        model = build_model(config, device)
        opt = torch.optim.Adam(model.parameters(), lr=config["training"]["learning_rate"])
        crit = torch.nn.CrossEntropyLoss()
        model.train()
        for _ in range(EPOCHS):
            for xb, yb in tl:
                xb, yb = xb.to(device), yb.to(device)
                opt.zero_grad(); loss = crit(model(xb), yb); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            preds = model(test_ds.X.to(device)).argmax(1).cpu().numpy()
        return (balanced_accuracy_score(y[te_idx], preds),
                f1_score(y[te_idx], preds, average="macro"))

    cnn_rand = quick_cnn(train_idx, test_idx)
    gss = GroupShuffleSplit(n_splits=1, test_size=TEST_RATIO, random_state=SEED)
    g_tr, g_te = next(gss.split(X, y, groups=subject))
    cnn_group = quick_cnn(g_tr, g_te)
    print(f"CNN random       bal-acc={cnn_rand[0]:.4f}  macro-F1={cnn_rand[1]:.4f}")
    print(f"CNN subject-grp  bal-acc={cnn_group[0]:.4f}  macro-F1={cnn_group[1]:.4f}")
    print(f"CNN leakage gap  bal-acc={cnn_rand[0] - cnn_group[0]:+.4f}")
else:
    print("RUN_CNN=False — skipping the slow CNN corroboration. "
          "The tree-model gap above already quantifies the leakage.")

## 8. Verdict

In [ ]:
gap = ba_rand - ba_group
signals = {
    "test-window subject overlap > 0.5": test_window_overlap > 0.5,
    "adjacent near-duplicate fraction > 0.05": dup_frac > 0.05,
    "random - grouped bal-acc gap > 0.10": gap > 0.10,
}
leakage_confirmed = any(signals.values())

mechanisms = []
if dup_frac > 0.05:
    mechanisms.append(f"overlapping near-duplicate windows across the split ({dup_frac:.1%} of test)")
if test_window_overlap > 0.5:
    mechanisms.append(f"subject leakage ({test_window_overlap:.1%} of test windows share a train subject)")
if src_auc.mean() > 0.9:
    mechanisms.append(f"trivially separable sources (OVR-AUC {src_auc.mean():.2f})")

print("=" * 70)
print("ADVERSARIAL VALIDATION — VERDICT")
print("=" * 70)
print(f"Adversarial train-vs-test AUC : {adv_auc.mean():.3f}  (~0.5 ⇒ no real held-out gap)")
print(f"Source-identification AUC     : {src_auc.mean():.3f}")
print(f"Test-window subject overlap   : {test_window_overlap:.3f}")
print(f"Adjacent near-duplicate frac  : {dup_frac:.3f}")
print(f"bal-acc random / grouped      : {ba_rand:.3f} / {ba_group:.3f}   gap = {gap:+.3f}")
print("-" * 70)
for name, hit in signals.items():
    print(f"  [{'X' if hit else ' '}] {name}")
print("-" * 70)
print(f"LEAKAGE CONFIRMED: {leakage_confirmed}")
if leakage_confirmed:
    print("Dominant mechanism(s):")
    for mech in mechanisms:
        print(f"  - {mech}")
    print(
        "\nRecommended fix:\n"
        "  Replace the random StratifiedShuffleSplit in\n"
        "  src/merged_dataset.py::load_and_prepare_data with a SUBJECT-GROUPED,\n"
        "  class-stratified split (StratifiedGroupKFold / GroupShuffleSplit on the\n"
        "  `subject` array, which the loader must read from the npz). This keeps every\n"
        "  subject — and therefore all its overlapping windows — entirely within one split,\n"
        "  eliminating both the near-duplicate and subject-identity leakage."
    )
print("=" * 70)